<a href="https://colab.research.google.com/github/roy2392/every-right-rag-solution/blob/main/KolZchut_crawler_%2B_vectorizing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 gspread google-auth google-auth-oauthlib google-auth-httplib2 openai pinecone-client

# **[OLD] Scrape and crawl all pages**
(Scrape all /he/ pages by ALL pages - without queue)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from collections import deque
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default

class WebCrawler:
    def __init__(self):
        self.base_url = 'https://www.kolzchut.org.il/he/'
        self.visited_urls = set()  # URLs we've already crawled
        self.queued_urls = set()   # For checking if URL is in queue
        self.urls_to_visit = deque()  # The actual queue
        self.pages_found = 0

        # Add the first URL
        self.add_to_queue(self.base_url)

        # Google Sheets setup remains the same
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)

        try:
            self.sheet = self.gc.open('KolZchut_Crawler_Data')
        except:
            self.sheet = self.gc.create('KolZchut_Crawler_Data')

        self.worksheet = self.sheet.get_worksheet(0)
        if not self.worksheet:
            self.worksheet = self.sheet.add_worksheet('Crawled Pages', 1000, 4)

        if not self.worksheet.row_values(1):
            self.worksheet.update('A1:D1', [['Title', 'URL', 'Last Crawl', 'Is Article']])

    def add_to_queue(self, url):
        """Add URL to queue only if it's not visited and not already queued"""
        if url not in self.visited_urls and url not in self.queued_urls:
            self.queued_urls.add(url)
            self.urls_to_visit.append(url)
            return True
        return False

    def remove_from_queue(self, url):
        """Remove URL from queue tracking"""
        self.queued_urls.remove(url)

    def is_valid_url(self, url):
        parsed = urlparse(url)
        return (parsed.netloc == 'www.kolzchut.org.il' and
                parsed.path.startswith('/he/'))

    def is_article(self, soup):
        if soup.find('span', class_='portal-boxes-table'):
          return False
        return True

    def add_to_spreadsheet(self, title, url, is_article):
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        self.worksheet.append_row([title, url, now, str(is_article)])

    def crawl_page(self, url):
        try:
            time.sleep(1)  # Respectful delay

            print(f"\n🔍 Currently crawling: {url}")

            response = requests.get(url, headers={
                'User-Agent': 'Mozilla/5.0 (compatible; Educational-Crawler)',
                'Accept-Encoding': 'utf-8'
            })
            response.encoding = 'utf-8'

            soup = BeautifulSoup(response.text, 'html.parser')

            # Get page title
            title = soup.title.string.strip() if soup.title else 'No Title'

            # Check if it's an article
            is_article = self.is_article(soup)

            # Mark as visited
            self.visited_urls.add(url)
            self.pages_found += 1

            # Add to spreadsheet
            self.add_to_spreadsheet(title, url, is_article)

            # Find all valid links on the page
            new_urls_found = 0
            for link in soup.find_all('a', href=True):
                next_url = urljoin(url, link['href'])
                if self.is_valid_url(next_url):
                    if self.add_to_queue(next_url):
                        new_urls_found += 1

            # Enhanced logging with accurate counts
            article_status = "📄 Article" if is_article else "🔗 Non-Article"
            print(f"{article_status} page")
            print(f"📈 Found {new_urls_found} new unique URLs on this page")
            print(f"\n📊 Progress Report:")
            print(f"  • Pages crawled: {self.pages_found}")
            print(f"  • Unique URLs visited: {len(self.visited_urls)}")
            print(f"  • Unique URLs in queue: {len(self.urls_to_visit)}")

        except Exception as e:
            print(f"❌ Error crawling {url}: {str(e)}")
        finally:
            # Always remove from queue tracking, even if there was an error
            self.remove_from_queue(url)

    def start_crawling(self, max_pages=None):
        print("🚀 Starting crawler...")
        print(f"📝 Data will be saved to Google Spreadsheet: 'KolZchut_Crawler_Data'")

        try:
            while self.urls_to_visit and (max_pages is None or self.pages_found < max_pages):
                next_url = self.urls_to_visit.popleft()
                self.crawl_page(next_url)

            print("\n✅ Crawling completed!")
            if not self.urls_to_visit:
                print("🎯 All discovered URLs have been crawled!")
            else:
                print(f"🛑 Reached maximum pages limit ({max_pages})")

            print(f"\n📈 Final Statistics:")
            print(f"  • Total pages crawled: {self.pages_found}")
            print(f"  • Total unique URLs visited: {len(self.visited_urls)}")
            print(f"  • Remaining URLs in queue: {len(self.urls_to_visit)}")

            # Get the sharing link for the spreadsheet
            sheet_url = self.sheet.url
            print(f"\n📊 You can view the results here: {sheet_url}")

        except KeyboardInterrupt:
            print("\n⚠️ Crawling interrupted by user")
            print(f"📈 Partial Statistics:")
            print(f"  • Pages crawled: {self.pages_found}")
            print(f"  • Unique URLs visited: {len(self.visited_urls)}")
            print(f"  • URLs remaining in queue: {len(self.urls_to_visit)}")

        return self.visited_urls

# Create and run the crawler
crawler = WebCrawler()
found_urls = crawler.start_crawling()

🚀 Starting crawler...
📝 Data will be saved to Google Spreadsheet: 'KolZchut_Crawler_Data'

⚠️ Crawling interrupted by user
📈 Partial Statistics:
  • Pages crawled: 0
  • Unique URLs visited: 0
  • URLs remaining in queue: 0


# **[OLD] Scrape all pages with another sheet for queue**
(Scrape all /he/ pages by ALL pages - with a "queue" sheet)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
import time
from collections import deque
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

class WebCrawler:
    def __init__(self):
        self.base_url = 'https://www.kolzchut.org.il/he/'
        self.visited_urls = set()
        self.queued_urls = set()
        self.urls_to_visit = deque()
        self.pages_found = 0
        self.queue_batch = []
        self.batch_size = 500
        self.crawled_batch = []
        self.crawled_batch_size = 500

        # Configure retry strategy
        self.session = requests.Session()
        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[500, 502, 503, 504]
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

        # Google Sheets setup
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)

        try:
            self.sheet = self.gc.open('KolZchut_Crawler_Data')
        except:
            self.sheet = self.gc.create('KolZchut_Crawler_Data')

        self.init_worksheets()
        self.load_previous_state()

    def init_worksheets(self):
        """Initialize worksheets with Last Crawl date column"""
        worksheets = self.sheet.worksheets()

        self.crawled_worksheet = None
        self.queue_worksheet = None

        for worksheet in worksheets:
            if worksheet.title == 'Crawled Pages':
                self.crawled_worksheet = worksheet
            elif worksheet.title == 'Queue':
                self.queue_worksheet = worksheet

        if not self.crawled_worksheet:
            self.crawled_worksheet = self.sheet.add_worksheet('Crawled Pages', 1000, 3)  # Added column for Last Crawl
            self.crawled_worksheet.update(values=[['Title', 'URL', 'Last Crawl']], range_name='A1:C1')

        if not self.queue_worksheet:
            self.queue_worksheet = self.sheet.add_worksheet('Queue', 1000, 4)
            self.queue_worksheet.update(values=[['URL', 'Status', 'Date Added to Queue', 'Date Processed']], range_name='A1:D1')

    def flush_batches(self):
        """Flush both queue and crawled batches to spreadsheet"""
        try:
            if self.queue_batch:
                self.queue_worksheet.append_rows(self.queue_batch)
                self.queue_batch = []

            if self.crawled_batch:
                self.crawled_worksheet.append_rows(self.crawled_batch)
                self.crawled_batch = []
        except Exception as e:
            print(f"Error flushing batches: {e}")
            time.sleep(60)  # Wait 60 seconds if we hit API limit
            self.flush_batches()  # Retry once after waiting

    def update_queue_status(self, url, status, date_processed=None):
        """Batch queue status updates"""
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        self.queue_batch.append([url, status, now, date_processed if date_processed else ''])

        if len(self.queue_batch) >= self.batch_size:
            self.flush_batches()

    def add_to_crawled(self, title, url):
        """Batch crawled pages updates with timestamp"""
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        self.crawled_batch.append([title, url, now])  # Added timestamp

        if len(self.crawled_batch) >= self.crawled_batch_size:
            self.flush_batches()

    def crawl_page(self, url):
        try:
            normalized_url = self.normalize_url(url)
            print(f"\n🔍 Currently crawling: {normalized_url}")

            try:
                response = self.session.get(normalized_url, headers={
                    'User-Agent': 'Mozilla/5.0 (compatible; Educational-Crawler)',
                    'Accept-Encoding': 'utf-8'
                })
                response.encoding = 'utf-8'
                response.raise_for_status()
            except requests.exceptions.RequestException as e:
                if '502' in str(e):
                    print(f"❌ 502 error for {normalized_url} after retries")
                    self.update_queue_status(normalized_url, '502')
                    return
                raise e

            soup = BeautifulSoup(response.text, 'html.parser')
            title = soup.title.string.strip() if soup.title else 'No Title'

            self.visited_urls.add(normalized_url)
            self.pages_found += 1

            self.add_to_crawled(title, normalized_url)

            new_urls_found = 0
            for link in soup.find_all('a', href=True):
                next_url = self.normalize_url(urljoin(url, link['href']))
                if self.is_valid_url(next_url):
                    if self.add_to_queue(next_url):
                        new_urls_found += 1

            print(f"📈 Found {new_urls_found} new unique URLs on this page")
            print(f"\n📊 Progress Report:")
            print(f"  • Pages crawled: {self.pages_found}")
            print(f"  • Unique URLs visited: {len(self.visited_urls)}")
            print(f"  • Unique URLs in queue: {len(self.urls_to_visit)}")

            now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            self.update_queue_status(normalized_url, 'crawled', now)

        except Exception as e:
            print(f"❌ Error crawling {normalized_url}: {str(e)}")
            self.update_queue_status(normalized_url, f'error: {str(e)}')
        finally:
            self.queued_urls.remove(normalized_url)

    def normalize_url(self, url):
        return urldefrag(url)[0]

    def load_previous_state(self):
        print("Loading previous state...")

        crawled_data = self.crawled_worksheet.get_all_values()[1:]
        for row in crawled_data:
            if row and len(row) >= 2:  # Changed from 3 to 2 since we only need URL
                self.visited_urls.add(self.normalize_url(row[1]))

        queue_data = self.queue_worksheet.get_all_values()[1:]
        for row in queue_data:
            if row and len(row) >= 2:
                url = self.normalize_url(row[0])
                status = row[1]
                if status == 'pending' and url not in self.visited_urls:
                    self.add_to_queue(url, update_sheet=False)

        if not self.urls_to_visit:
            self.add_to_queue(self.base_url)
            self.flush_batches()

        print(f"Loaded {len(self.visited_urls)} crawled URLs and {len(self.urls_to_visit)} pending URLs")

    def add_to_queue(self, url, update_sheet=True):
        normalized_url = self.normalize_url(url)
        if normalized_url not in self.visited_urls and normalized_url not in self.queued_urls:
            self.queued_urls.add(normalized_url)
            self.urls_to_visit.append(normalized_url)
            if update_sheet:
                self.update_queue_status(normalized_url, 'pending')
            return True
        return False

    def is_valid_url(self, url):
        parsed = urlparse(url)
        return (parsed.netloc == 'www.kolzchut.org.il' and
                parsed.path.startswith('/he/'))

    def start_crawling(self, max_pages=None):
        print("🚀 Starting crawler...")
        print(f"📝 Data will be saved to Google Spreadsheet: 'KolZchut_Crawler_Data'")

        try:
            while self.urls_to_visit and (max_pages is None or self.pages_found < max_pages):
                next_url = self.urls_to_visit.popleft()
                self.crawl_page(next_url)

            self.flush_batches()

            print("\n✅ Crawling completed!")
            if not self.urls_to_visit:
                print("🎯 All discovered URLs have been crawled!")
            else:
                print(f"🛑 Reached maximum pages limit ({max_pages})")

            print(f"\n📈 Final Statistics:")
            print(f"  • Total pages crawled: {self.pages_found}")
            print(f"  • Total unique URLs visited: {len(self.visited_urls)}")
            print(f"  • Remaining URLs in queue: {len(self.urls_to_visit)}")

            sheet_url = self.sheet.url
            print(f"\n📊 You can view the results here: {sheet_url}")

        except KeyboardInterrupt:
            print("\n⚠️ Crawling interrupted by user")
            print("💾 Flushing remaining items...")
            self.flush_batches()
            print(f"📈 Partial Statistics:")
            print(f"  • Pages crawled: {self.pages_found}")
            print(f"  • Unique URLs visited: {len(self.visited_urls)}")
            print(f"  • URLs remaining in queue: {len(self.urls_to_visit)}")

        return self.visited_urls

# Create and run the crawler
crawler = WebCrawler()
found_urls = crawler.start_crawling()

Streaming output truncated to the last 5000 lines.
🔍 Currently crawling: http://www.kolzchut.org.il/he/%D7%A7%D7%A8%D7%9F_%D7%A4%D7%A0%D7%A1%D7%99%D7%94
📈 Found 0 new unique URLs on this page

📊 Progress Report:
  • Pages crawled: 13923
  • Unique URLs visited: 14550
  • Unique URLs in queue: 24928

🔍 Currently crawling: http://www.kolzchut.org.il/he/%D7%91%D7%99%D7%98%D7%95%D7%97_%D7%9E%D7%A0%D7%94%D7%9C%D7%99%D7%9D
📈 Found 0 new unique URLs on this page

📊 Progress Report:
  • Pages crawled: 13924
  • Unique URLs visited: 14551
  • Unique URLs in queue: 24927

🔍 Currently crawling: http://www.kolzchut.org.il/he/%D7%A7%D7%95%D7%A4%D7%AA_%D7%92%D7%9E%D7%9C_%D7%9C%D7%94%D7%A9%D7%A7%D7%A2%D7%94
📈 Found 0 new unique URLs on this page

📊 Progress Report:
  • Pages crawled: 13925
  • Unique URLs visited: 14552
  • Unique URLs in queue: 24926

🔍 Currently crawling: http://www.kolzchut.org.il/he/%D7%AA%D7%95%D7%9B%D7%A0%D7%99%D7%AA_%22%D7%97%D7%99%D7%A1%D7%9B%D7%95%D7%9F_%D7%9C%D7%9B%D7%9C_%D

# **1. Scarpe all portal pages**
Write to the "All Portals" sheet all links in this page:

www.kolzchut.org.il/he/כל-זכות:פורטלים_פעילים

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default

class PortalExtractor:
    def __init__(self):
        # URL of the portals page
        self.url = 'https://www.kolzchut.org.il/he/%D7%9B%D7%9C-%D7%96%D7%9B%D7%95%D7%AA:%D7%A4%D7%95%D7%A8%D7%98%D7%9C%D7%99%D7%9D_%D7%A4%D7%A2%D7%99%D7%9C%D7%99%D7%9D'

        # Initialize Google Sheets connection
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)

        # Open existing spreadsheet
        try:
            self.sheet = self.gc.open('KolZchut_Crawler_Data')
        except:
            print("❌ Error: Could not find 'KolZchut_Crawler_Data' spreadsheet")
            raise

        self.setup_worksheet()

    def setup_worksheet(self):
        """Initialize the worksheet with headers"""
        # Check if worksheet exists
        try:
            self.worksheet = self.sheet.worksheet('All portals')
        except:
            self.worksheet = self.sheet.add_worksheet('All portals', 1000, 3)

        # Set headers if worksheet is empty
        if self.worksheet.row_count == 0:
            headers = [['Portal name', 'Portal URL', 'Date added']]
            self.worksheet.update('A1:C1', headers)

    def extract_portals(self):
        """Extract portal links from the CategoryTreeTag div"""
        try:
            # Make request to the page
            response = requests.get(self.url, headers={
                'User-Agent': 'Mozilla/5.0 (compatible; Educational-Script)',
                'Accept-Encoding': 'utf-8'
            })
            response.encoding = 'utf-8'
            response.raise_for_status()

            # Parse the HTML
            soup = BeautifulSoup(response.text, 'html.parser')

            # Find the CategoryTreeTag div
            category_tree = soup.find('div', class_='CategoryTreeTag')

            if not category_tree:
                raise Exception("CategoryTreeTag div not found")

            # Extract all links
            portal_data = []
            current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            for link in category_tree.find_all('a'):
                portal_name = link.text.strip()
                portal_url = 'https://www.kolzchut.org.il' + link.get('href')
                portal_data.append([portal_name, portal_url, current_time])

            # Update spreadsheet
            if portal_data:
                self.worksheet.append_rows(portal_data)
                print(f"✅ Successfully extracted {len(portal_data)} portals")
                print(f"📊 Data saved to spreadsheet: {self.sheet.url}")
                print(f"📑 Check the 'All portals' worksheet")
            else:
                print("⚠️ No portal links found")

        except Exception as e:
            print(f"❌ Error: {str(e)}")

def main():
    extractor = PortalExtractor()
    extractor.extract_portals()

if __name__ == "__main__":
    main()

✅ Successfully extracted 312 portals
📊 Data saved to spreadsheet: https://docs.google.com/spreadsheets/d/1-YbhieKH7Mn3NG3cnOF1E78EjSXAhUiciS2-WnUcGR4
📑 Check the 'All portals' worksheet


# **2. Crawl all portals in "All Portals" sheet**
Crawl each portal page by reading all /he/ links in the "portal-box" divs and write them to the "Crawled Pages" sheet.

In [ ]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
from google.colab import auth
import gspread
from google.auth import default
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from urllib.parse import urlparse

class PortalContentCrawler:
    def __init__(self):
        # Initialize Google Sheets connection
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)

        # Configure retry strategy for requests
        self.session = requests.Session()
        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[500, 502, 503, 504]
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)

        # Open existing spreadsheet
        try:
            self.sheet = self.gc.open('KolZchut_Crawler_Data')
            self.portals_sheet = self.sheet.worksheet('All portals')
            self.crawled_sheet = self.sheet.worksheet('Crawled Pages')
        except Exception as e:
            print(f"❌ Error accessing spreadsheet: {str(e)}")
            raise

        # Initialize batch for crawled pages
        self.crawled_batch = []
        self.batch_size = 100

    def is_valid_url(self, url):
        """Check if URL is part of kolzchut.org.il/he/"""
        try:
            parsed = urlparse(url)
            return (parsed.netloc == 'www.kolzchut.org.il' and
                   parsed.path.startswith('/he/'))
        except:
            return False

    def normalize_url(self, base_url, href):
        """Normalize relative URLs to absolute URLs"""
        if href.startswith('/'):
            return 'https://www.kolzchut.org.il' + href
        return href

    def get_portal_data(self):
        """Get all portal URLs and names from the All portals sheet"""
        data = self.portals_sheet.get_all_values()
        portal_data = data[1:]
        return [(row[0], row[1]) for row in portal_data if len(row) >= 2]

    def crawl_portal_page(self, portal_name, portal_url):
        """Crawl a portal page and extract links from portal-box divs"""
        try:
            print(f"\n🔍 Crawling portal: {portal_name}")

            response = self.session.get(portal_url, headers={
                'User-Agent': 'Mozilla/5.0 (compatible; Educational-Crawler)',
                'Accept-Encoding': 'utf-8'
            })
            response.encoding = 'utf-8'
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'html.parser')
            portal_boxes = soup.find_all('div', class_='portal-box')

            if not portal_boxes:
                print(f"⚠️ No portal-box divs found in {portal_name}")
                return 0

            links_found = 0
            invalid_links = 0
            current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            for box in portal_boxes:
                for link in box.find_all('a'):
                    title = link.text.strip()
                    url = self.normalize_url(portal_url, link.get('href', ''))

                    if self.is_valid_url(url):
                        self.crawled_batch.append([portal_name, title, url, current_time])
                        links_found += 1

                        if len(self.crawled_batch) >= self.batch_size:
                            self.flush_batch()
                    else:
                        invalid_links += 1

            print(f"📈 Found {links_found} valid links in {portal_name}")
            if invalid_links > 0:
                print(f"ℹ️ Skipped {invalid_links} invalid links")
            return links_found

        except Exception as e:
            print(f"❌ Error crawling {portal_url}: {str(e)}")
            return 0

    def flush_batch(self):
        """Flush the batch of crawled pages to the spreadsheet"""
        try:
            if self.crawled_batch:
                self.crawled_sheet.append_rows(self.crawled_batch)
                print(f"💾 Saved batch of {len(self.crawled_batch)} entries")
                self.crawled_batch = []
        except Exception as e:
            print(f"❌ Error saving batch: {str(e)}")
            time.sleep(60)  # Wait if we hit API limit
            self.flush_batch()  # Retry once after waiting

    def start_crawling(self):
        """Start the crawling process"""
        print("🚀 Starting portal content crawler...")

        try:
            portals = self.get_portal_data()
            total_portals = len(portals)
            total_links = 0

            print(f"📋 Found {total_portals} portals to crawl")

            for index, (portal_name, portal_url) in enumerate(portals, 1):
                print(f"\n📌 Processing portal {index}/{total_portals}")
                links_found = self.crawl_portal_page(portal_name, portal_url)
                total_links += links_found

                # Add small delay between requests
                time.sleep(1)

            # Flush any remaining items in batch
            self.flush_batch()

            print("\n✅ Crawling completed!")
            print(f"📊 Final Statistics:")
            print(f"  • Portals processed: {total_portals}")
            print(f"  • Total valid links found: {total_links}")
            print(f"  • Results saved to: {self.sheet.url}")

        except KeyboardInterrupt:
            print("\n⚠️ Crawling interrupted by user")
            print("💾 Saving remaining items...")
            self.flush_batch()

        except Exception as e:
            print(f"❌ Error during crawling: {str(e)}")
            self.flush_batch()

def main():
    crawler = PortalContentCrawler()
    crawler.start_crawling()

if __name__ == "__main__":
    main()

🚀 Starting portal content crawler...
📋 Found 311 portals to crawl

📌 Processing portal 1/311

🔍 Crawling portal: אבחונים לתלמידי בית הספר
📈 Found 10 valid links in אבחונים לתלמידי בית הספר

📌 Processing portal 2/311

🔍 Crawling portal: אבטלה וזכויות מובטלים
📈 Found 47 valid links in אבטלה וזכויות מובטלים
ℹ️ Skipped 5 invalid links

📌 Processing portal 3/311

🔍 Crawling portal: אביזרי ומכשירי שיקום
📈 Found 36 valid links in אביזרי ומכשירי שיקום

📌 Processing portal 4/311

🔍 Crawling portal: אגרת רשות השידור (אגרת טלוויזיה)
💾 Saved batch of 100 entries
📈 Found 15 valid links in אגרת רשות השידור (אגרת טלוויזיה)

📌 Processing portal 5/311

🔍 Crawling portal: אוויר נקי ומניעת זיהום אוויר
📈 Found 17 valid links in אוויר נקי ומניעת זיהום אוויר
ℹ️ Skipped 3 invalid links

📌 Processing portal 6/311

🔍 Crawling portal: אומנה
📈 Found 52 valid links in אומנה
ℹ️ Skipped 1 invalid links

📌 Processing portal 7/311

🔍 Crawling portal: אזרחים ותיקים ופנסיונרים (גיל הזהב)
💾 Saved batch of 100 entries
📈 

# **3. Chunk and vectorize pages in spreadsheet**
Go over all "approve_vector" = "yes" items in the "Crawled Pages" sheet, break it to chunks, send each chunk to OpenAI embeddings (ada-002), and then send each chunk to Pinecone vector database.

Each row in the "Crawled Pages" item get the number of chunks sent to Pinecone.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from google.colab import auth
import gspread
from google.auth import default
import time
from openai import OpenAI
import os
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec
import uuid
from datetime import datetime
import pandas as pd


client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Initialize the Pinecone client
pc = Pinecone(
    api_key=userdata.get("PINECONE_API_KEY")
)
# Connect to your index
index = pc.Index('kolzchut-colab')


class ContentProcessor:

    def get_openai_embedding(self, chunk, title):
        """Get OpenAI vector embedding for the given text chunk"""
        try:
            # Create the context text
            context_text = f"[Context]\nThis is a chunk of a larger article.\nThe article title is: \"{title}\"\n[End context]\n{chunk}"

            response = client.embeddings.create(
                model="text-embedding-ada-002",
                input=context_text  # Use context_text instead of chunk
            )
            embedding = response.data[0].embedding
            return embedding
        except Exception as e:
            print(f"❌ Error generating embedding: {str(e)}")
            return None


    def __init__(self):
        # Authenticate and get spreadsheet
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)
        self.sheet = self.gc.open('KolZchut_Crawler_Data')

        # Get the "Crawled Pages" worksheet specifically
        try:
            self.worksheet = self.sheet.worksheet('Crawled Pages')

            # Ensure required columns exist
            required_columns = ['Title', 'URL', 'Last Crawl', 'approve_vector', 'recent_vectorize', 'chunks']
            headers = self.worksheet.row_values(1)

            # Add missing columns if needed
            for column in required_columns:
                if column not in headers:
                    next_col = len(headers) + 1
                    self.worksheet.update_cell(1, next_col, column)
                    headers.append(column)
                    print(f"Added missing column: {column}")

        except gspread.exceptions.WorksheetNotFound:
            print("❌ Error: 'Crawled Pages' worksheet not found!")
            raise

        # Define elements to exclude
        self.exclude_tags = ['header', 'footer']
        self.exclude_class_contains = ['nav', 'cookie', 'menu']
        self.exclude_classes = ['toc-box', 'article-bottom', 'article-see-also']
        self.exclude_ids = ['helpme-section']

        # Chunking parameters
        self.chunk_size = 1900
        self.overlap_size = 100

    def should_exclude_element(self, element):
        """Check if an element should be excluded based on our criteria"""
        try:
            # Check tag name
            if element.name in self.exclude_tags:
                return True

            # Get element's classes as a list
            classes = element.get('class', [])
            if not isinstance(classes, (list, tuple)):
                classes = [classes]

            # Check class contains
            for class_name in classes:
                if isinstance(class_name, str):  # Make sure class_name is a string
                    if any(exclude in class_name.lower() for exclude in self.exclude_class_contains):
                        return True

            # Check specific classes
            if any(class_name in self.exclude_classes for class_name in classes if isinstance(class_name, str)):
                return True

            # Check ID
            element_id = element.get('id', '')
            if element_id in self.exclude_ids:
                return True

            return False
        except Exception as e:
            return False

    def extract_clean_text(self, url):
        """Extract and clean text from URL"""
        try:
            # Make the request with proper headers
            response = requests.get(
                url,
                headers={
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
                    'Accept-Language': 'he,en-US;q=0.7,en;q=0.3',
                    'Accept-Encoding': 'gzip, deflate, br'
                },
                timeout=10
            )
            response.raise_for_status()
            response.encoding = 'utf-8'

            soup = BeautifulSoup(response.text, 'html.parser')

            for element in soup.find_all():
                if self.should_exclude_element(element):
                    element.decompose()

            text = soup.get_text(separator='\n', strip=True)
            text = re.sub(r'\s+\n\s+', '\n', text)
            text = re.sub(r'\n+', '\n', text)
            text = text.strip()

            print(f"📝 Extracted {len(text)} characters of text")
            return text

        except requests.RequestException as e:
            print(f"❌ Network error while fetching {url}: {str(e)}")
            return None
        except Exception as e:
            print(f"❌ Error extracting text from {url}: {str(e)}")
            return None

    def create_chunks(self, text):
        """Create overlapping chunks from text"""
        if not text:
            return []

        chunks = []
        start = 0
        total_text_length = len(text)

        print(f"📏 Total text length: {total_text_length}")

        while start < total_text_length:
            end = start + self.chunk_size

            if end > total_text_length:
                end = total_text_length

            chunk = text[start:end]

            if len(chunk) > 100:
                chunks.append(chunk)
                print(f"✅ Added chunk: {len(chunk)} characters")

            if len(text[start:]) <= 100:
                break

            start = end - self.overlap_size

        return chunks

    def process_content(self):
        print("🚀 Starting content processing...")

        try:
            # Get all data from the worksheet
            all_data = self.worksheet.get_all_records()

            if not all_data:
                print("❌ No data found in the worksheet!")
                return

            total_processed = 0
            total_failed = 0

            # Get column indexes (adding 1 because spreadsheet is 1-based)
            recent_vectorize_col = self.worksheet.find('recent_vectorize').col
            chunks_col = self.worksheet.find('chunks').col

            for row_num, row in enumerate(all_data, start=2):  # start=2 to account for header row
                try:
                    # Check if approved for vectorization
                    if row.get('approve_vector', '').lower() != 'yes':
                        continue

                    url = row.get('URL', '')  # Changed from 'url' to 'URL' to match the sheet
                    title = row.get('Title', 'untitled')  # Changed from 'title' to 'Title'

                    if not url:
                        print(f"⚠️ Skipping row {row_num}: No URL found")
                        continue

                    print(f"\n{'='*80}")
                    print(f"📄 Processing: {title}")
                    print(f"🔗 URL: {url}")

                    text = self.extract_clean_text(url)
                    if not text:
                        print("❌ Failed to extract text, skipping...")
                        total_failed += 1
                        continue

                    chunks = self.create_chunks(text)

                    for i, chunk in enumerate(chunks, 1):
                        print(f"\nChunk {i}/{len(chunks)}:")
                        print(f"Characters: {len(chunk)}")
                        print(f"Content preview: {chunk}")

                        embedding = self.get_openai_embedding(chunk, title)
                        if embedding:
                            vector_id = str(uuid.uuid4())
                            context_text = f"[Context]\nThis is a chunk of a larger article.\nThe article title is: \"{title}\"\n[End context]\n{chunk}"
                            metadata = {
                                "page_title": title,
                                "url": url,
                                "text": context_text
                            }

                            index.upsert(vectors=[(vector_id, embedding, metadata)])
                            print(f"✅ Upserted chunk {i} with ID: {vector_id} to Pinecone")

                    # Update the spreadsheet with timestamps and chunk count
                    if chunks:
                        self.worksheet.update_cell(row_num, recent_vectorize_col,
                                                 datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                        self.worksheet.update_cell(row_num, chunks_col, len(chunks))
                        total_processed += 1
                    else:
                        print("⚠️ No valid chunks created (text might be too short)")
                        total_failed += 1

                    # Add a small delay to avoid rate limits
                    time.sleep(1)

                except Exception as e:
                    print(f"❌ Error processing row {row_num}: {str(e)}")
                    total_failed += 1
                    continue

            print(f"\n✅ Processing completed!")
            print(f"📈 Summary:")
            print(f"  • Successfully processed: {total_processed} pages")
            print(f"  • Failed to process: {total_failed} pages")
            print(f"  • Total attempted: {total_processed + total_failed} pages")

        except Exception as e:
            print(f"❌ Critical error during processing: {str(e)}")

# Create and run the processor
processor = ContentProcessor()
processor.process_content()

Streaming output truncated to the last 5000 lines.
בנוסף, זכאי העובד להגיש תביעה לפיצויים אזרחיים לפי החוק להגנת הפרטיות (לרבות פיצויים ללא הוכחת נזק) בבית המשפט השלום.
פגיעה בפרטיות נחשבת גם עבירה פלילית והעובד רשאי להגיש תלונה נגד המעסיק במשטרה. במקרה זה עשוי המעסיק להיות חשוף להליכים פליליים או מנהליים.
גורמים מסייעים
ארגוני סיוע
לרשימת ארגונים המסייעים בתחום התעסוקה
גורמי ממשל
גורם ממשלתי
תחומי אחריות
נושאים
נציבות שוויון הזדמנויות בעבודה
הנציבות אמונה על אכיפת כל הקשור בשוויון בעבודה ובאכיפת מקרים בהם התקיימה אפליה אסורה
שוויון הזדמנויות בעבודה
,
נשים עובדות
הממונה על חוק עבודת נשים
נציבות זו ממונה על אכיפת
חוק עבודת נשים
שוויון הזדמנויות בעבודה
,
נשים עובדות
משרד העבודה
אחראית על כל התחומים הקשורים לתעסוקה
תעסוקה וזכויות עובדים
המוסד לבטיחות ולגיהות
מייעץ ומסייע בתחומים של קידום הבטיחות והבריאות התעסוקתית
סביבת עבודה בטוחה, תנאי העבודה
מקורות משפטיים ורשמיים
פסקי דין
מעסיק אינו רשאי לחדור לתכתובת דוא
✅ Upserted chunk 3 with ID: 45dcf759-c699-4236-9f76-bc2fa104d517 to Pinecone

Ch